# Marketing A/B test (YSDA A/B Week toolkit)

Dataset: Kaggle *Marketing A/B Test — Synthetic (Known Uplift)* (`data/ab_test.csv`).

Primary metric: user-level conversion. Methods follow the ШАД A/B Week notebooks: t/z tests, Mann–Whitney, Poisson bootstrap, delta method, linearization, bucketing, CUPED, post-stratification, MDE, AA checks, Holm correction.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_dataset, load_ground_truth, split_groups
from src.stats import (
    aa_pvalue_uniformity,
    bucket_ratio,
    cuped,
    delta_method_ratio,
    linearize_ratio,
    mann_whitney,
    mde_for_sample_size,
    poisson_bootstrap_mean_diff,
    srm_chi2,
    stratified_mean_diff,
    two_proportion_ztest,
    uplift_mean,
    welch_ttest,
)

df = load_dataset()
gt = load_ground_truth()
df.head()

## EDA and SRM

In [ ]:
print(df.groupby("group").size())
print(df.groupby("group")["converted"].mean())
test, control = split_groups(df)
print(srm_chi2(len(test), len(control)))

## ATE: t-test, z-test, Mann–Whitney, bootstrap, relative uplift

In [ ]:
for res in [
    welch_ttest(test, control),
    two_proportion_ztest(test, control),
    mann_whitney(test, control),
    poisson_bootstrap_mean_diff(test, control, n_iterations=1000),
    uplift_mean(test, control),
]:
    print(res)
print("true ATE pp", gt["true_average_treatment_effect_pp"])

## Ratio methods (conversion as sum(converted)/sum(users))

In [ ]:
ones_t, ones_c = np.ones(len(test)), np.ones(len(control))
for res in [
    delta_method_ratio(test, ones_t, control, ones_c),
    linearize_ratio(test, ones_t, control, ones_c, kind=1),
    linearize_ratio(test, ones_t, control, ones_c, kind=2),
    bucket_ratio(test, ones_t, control, ones_c),
]:
    print(res)

## CUPED and post-stratification

In [ ]:
print(cuped(df.converted.to_numpy(float), df.prior_purchases.to_numpy(float), df.treatment.to_numpy()))
print(stratified_mean_diff(df, "converted", "device"))
print("MDE pp", 100 * mde_for_sample_size(df.converted.std(ddof=1), len(df)))

## AA check and segment CATE

In [ ]:
aa = aa_pvalue_uniformity(df.converted.to_numpy(float), n_sim=200)
print({k: aa[k] for k in aa if k != "pvalues"})

for col in ["device", "is_new", "is_zero_prior"]:
    print("\n", col)
    for key, part in df.groupby(col):
        t, c = split_groups(part)
        r = welch_ttest(t, c)
        print(key, f"ATE={100*r.estimate:.2f}pp", f"p={r.pvalue:.3g}")